# 🧬 BioEmu — Protein Structure Sampling (v3)
> **Step 1:** `Runtime → Change runtime type → T4 GPU`  
> **Step 2:** Run cells top to bottom, one at a time.
---

## Cell 1 — Install BioEmu inside a Python 3.10 virtualenv
This cell creates a self-contained environment that bypasses Colab's system Python entirely.
**Takes ~10 min.** Do not skip or restart mid-way.

In [2]:
import subprocess, sys, os

VENV = '/content/bioemu_env'

# 1. Add deadsnakes PPA (has Python 3.10 for Ubuntu 24)
print('📦 Adding deadsnakes PPA...')
subprocess.run(['apt-get', 'install', '-y', '-qq', 'software-properties-common'], check=True)
subprocess.run(['add-apt-repository', '-y', 'ppa:deadsnakes/ppa'], check=True)
subprocess.run(['apt-get', 'update', '-qq'], check=True)
print('✅ PPA added')

# 2. Install Python 3.10
print('📦 Installing Python 3.10...')
subprocess.run(['apt-get', 'install', '-y', '-qq',
                'python3.10', 'python3.10-venv', 'python3.10-dev'], check=True)
print('✅ Python 3.10 ready')

# 3. Create virtualenv
print('🔧 Creating virtualenv...')
subprocess.run(['python3.10', '-m', 'venv', VENV], check=True)
PIP = f'{VENV}/bin/pip'
print('✅ Virtualenv created')

# 4. Install BioEmu inside the venv
print('⬇️  Installing BioEmu (~10 min, please wait)...')
subprocess.run([PIP, 'install', '--quiet', 'bioemu[cuda]'], check=True)
print('✅ BioEmu installed!')

# 5. Verify
result = subprocess.run(
    [f'{VENV}/bin/python', '-c', 'import bioemu; print("bioemu import OK")'],
    capture_output=True, text=True
)
print(result.stdout.strip() or result.stderr.strip())

📦 Adding deadsnakes PPA...
✅ PPA added
📦 Installing Python 3.10...
✅ Python 3.10 ready
🔧 Creating virtualenv...
✅ Virtualenv created
⬇️  Installing BioEmu (~10 min, please wait)...
✅ BioEmu installed!
bioemu import OK


## Cell 2 — Set your protein sequence
✏️ Edit `SEQUENCE` here. Default is chignolin (10 residues) — fast test run.

In [11]:
# ✏️ EDIT THESE:
SEQUENCE    = 'MTNLQDQTQQIVPFIRSLLMPTTGPASIPDDTLEKHTLRSETSTYNLTVGDTGSGLIVFFPGFPGSIVGAHYTLQSNGNYKFDQMLLTAQNLPASYNYCRLVSRSLTVRSSTLPGGVYALNGTINAVTFQGSLSELTDVSYNGLMSATANINDKIGNVLVGEGVTVLSLPTSYDLGYVRLGDPIPAIGLDPKMVATCDSSDRPRVYTITAADDYQFSSQYQAGGVTITLFSANIDAITSLSIGGELVFQTSVQGLILGATIYLIGFDGTAVITRAVAADNGLTAGTDNLMPFNIVIPTSEITQPITSIKLEIVTSKSGGQAGDQMSWSASGSLAVTIHGGNYPGALRPVTLVAYERVATGSVVTVAGVSNFELIPNPELAKNLVTEYGRFDPGAMNYTKLILSERDRLGIKTVWPTREYTDFREYFMEVADLNSPLKIAGA'          # your protein sequence here
NUM_SAMPLES = 10
OUTPUT_DIR  = '/content/bioemu_output'

L = len(SEQUENCE)
if   L <= 100: est = '~5–15 min on T4'
elif L <= 300: est = '~1–2 h on T4'
else:          est = '~4+ h on T4 — consider fewer samples'

print(f'Sequence : {SEQUENCE}')
print(f'Length   : {L} residues')
print(f'Samples  : {NUM_SAMPLES}')
print(f'Est. time: {est}')

Sequence : MTNLQDQTQQIVPFIRSLLMPTTGPASIPDDTLEKHTLRSETSTYNLTVGDTGSGLIVFFPGFPGSIVGAHYTLQSNGNYKFDQMLLTAQNLPASYNYCRLVSRSLTVRSSTLPGGVYALNGTINAVTFQGSLSELTDVSYNGLMSATANINDKIGNVLVGEGVTVLSLPTSYDLGYVRLGDPIPAIGLDPKMVATCDSSDRPRVYTITAADDYQFSSQYQAGGVTITLFSANIDAITSLSIGGELVFQTSVQGLILGATIYLIGFDGTAVITRAVAADNGLTAGTDNLMPFNIVIPTSEITQPITSIKLEIVTSKSGGQAGDQMSWSASGSLAVTIHGGNYPGALRPVTLVAYERVATGSVVTVAGVSNFELIPNPELAKNLVTEYGRFDPGAMNYTKLILSERDRLGIKTVWPTREYTDFREYFMEVADLNSPLKIAGA
Length   : 441 residues
Samples  : 10
Est. time: ~4+ h on T4 — consider fewer samples


## Cell 3 — Run BioEmu sampling
Runs inside the Python 3.10 venv via subprocess — no Python version conflict.

In [22]:
import subprocess, os

VENV = '/content/bioemu_env'
PIP = f'{VENV}/bin/pip'

print('🔧 Pinning JAX + Haiku to compatible versions...')
result = subprocess.run(
    [PIP, 'install',
     'jax[cuda12]==0.4.35',
     'jaxlib==0.4.34',       # ← correct version
     'dm-haiku==0.0.12',
    ],
    capture_output=True, text=True
)
print(result.stdout[-1000:])
print(result.stderr[-500:])

# Verify
r = subprocess.run(
    [f'{VENV}/bin/python', '-c',
     'import jax, haiku; print(f"jax {jax.__version__}, haiku {haiku.__version__}")'],
    capture_output=True, text=True
)
print(r.stdout.strip() or r.stderr[-300:])

🔧 Pinning JAX + Haiku to compatible versions...
 (from markdown-it-py>=2.2.0->rich>=11.1->flax>=0.7.1->dm-haiku==0.0.12) (0.1.2)
  Attempting uninstall: dm-haiku
    Found existing installation: dm-haiku 0.0.17
    Uninstalling dm-haiku-0.0.17:
      Successfully uninstalled dm-haiku-0.0.17


jax 0.4.35, haiku 0.0.12


In [23]:
import subprocess, os, shutil, math

OUTPUT_DIR = '/content/bioemu_output'
VENV = '/content/bioemu_env'

shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

L = len(SEQUENCE)
batch_size_100 = math.ceil((L / 100) ** 2)

script = f"""
from bioemu.sample import main
main(
    sequence='{SEQUENCE}',
    num_samples={NUM_SAMPLES},
    output_dir='{OUTPUT_DIR}',
    batch_size_100={batch_size_100},
)
"""

# Fix: clear the bad matplotlib backend env var before launching subprocess
env = os.environ.copy()
env.pop('MPLBACKEND', None)           # ← the fix

process = subprocess.Popen(
    [f'{VENV}/bin/python', '-c', script],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, env=env                # ← pass clean env
)
print(f'▶️  Sampling {L}-residue protein (batch_size_100={batch_size_100})...\n')
for line in process.stdout:
    print(line, end='')
process.wait()

if process.returncode == 0:
    print(f'\n✅ Done! Files in {OUTPUT_DIR}')
else:
    print(f'\n❌ Exited with code {process.returncode}')

▶️  Sampling 441-residue protein (batch_size_100=20)...

/content/bioemu_env/lib/python3.10/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(

Sampling batches...:   0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/150 [elapsed: 00:00 remaining: ?]

SUBMIT:   0%|          | 0/150 [elapsed: 00:00 remaining: ?]

COMPLETE:   0%|          | 0/150 [elapsed: 00:00 remaining: ?]

COMPLETE: 100%|██████████| 150/150 [elapsed: 00:00 remaining: 00:00]
I0000 00:00:1790241283.429417   14714 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.









































































































































In [17]:
# Inspect BioEmu's actual batch size formula
cmd = [f'{VENV}/bin/python', '-c', '''
import inspect
from bioemu.sample import main
src = inspect.getsource(main)
# Print the batch_size calculation lines
for i, line in enumerate(src.split("\\n")):
    if "batch_size" in line.lower():
        print(f"{i:3d}: {line}")
''']
import subprocess
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout)
print(r.stderr[-500:] if r.stderr else '')

  6:     batch_size_100: int = 10,
 27:         batch_size_100: Batch size you'd use for a sequence of length 100. The batch size will be calculated from this, assuming
169:     batch_size = int(batch_size_100 * (100 / len(sequence)) ** 2)
171:     batch_size = min(batch_size, num_samples)
172:     logger.info(f"Using batch size {min(batch_size, num_samples)}")
177:         range(existing_num_samples, num_samples, batch_size), desc="Sampling batches..."
179:         n = min(batch_size, num_samples - start_idx)
192:             batch_size=min(batch_size, n),

/content/bioemu_env/lib/python3.10/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(



## Cell 4 — List output files

In [24]:
import os
OUTPUT_DIR = '/content/bioemu_output'
files = sorted(os.listdir(OUTPUT_DIR))
print(f'{len(files)} output file(s):')
for f in files:
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f:45s}  {size/1024:.1f} KB')

13 output file(s):
  batch_0000000_0000001.npz                      23.2 KB
  batch_0000001_0000002.npz                      23.2 KB
  batch_0000002_0000003.npz                      23.2 KB
  batch_0000003_0000004.npz                      23.2 KB
  batch_0000004_0000005.npz                      23.2 KB
  batch_0000005_0000006.npz                      23.2 KB
  batch_0000006_0000007.npz                      23.2 KB
  batch_0000007_0000008.npz                      23.2 KB
  batch_0000008_0000009.npz                      23.2 KB
  batch_0000009_0000010.npz                      23.2 KB
  samples.xtc                                    16.4 KB
  sequence.fasta                                 0.5 KB
  topology.pdb                                   171.1 KB


## Cell 5 — Download results as ZIP

In [25]:
import shutil
from google.colab import files

OUTPUT_DIR = '/content/bioemu_output'
zip_path   = '/content/bioemu_results'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
files.download(zip_path + '.zip')
print('📦 Download started!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📦 Download started!


---
## Troubleshooting
| Error | Fix |
|---|---|
| `python3.10: command not found` | Re-run Cell 1 |
| `CUDA out of memory` | Reduce `NUM_SAMPLES` or sequence length |
| Session disconnects | Colab free tier ~90 min idle limit; re-run Cell 3 only |
| `No such file bioemu_output` | Run Cell 3 first |

**Citation:** Lewis et al., *Science* 2025, doi:10.1126/science.adv9817